In [0]:
%pip install pypdf langchain databricks-langchain
dbutils.library.restartPython()

In [0]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
pdf_path = "Nvidia_Wikipedia.pdf"
reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += page.extract_text() or ""

# Chunk text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", " "]
)
chunks = text_splitter.split_text(text)

print(f"✅ Created {len(chunks)} chunks.")
print(chunks[0][:500])  # Preview first chunk

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",  # Replace with your model endpoint
    temperature=0.1,
    max_tokens=512
)

In [0]:
import warnings
import time
from typing import Any

warnings.filterwarnings("ignore", category=UserWarning)

summaries = []
max_chunks_to_process = 10 
rate_limit_delay = 1.0

for i, chunk in enumerate(chunks[:max_chunks_to_process]):
    prompt = f"Summarize text chunk {i+1} of {max_chunks_to_process}:\n\n{chunk}"

combined_text = "\n".join(summaries)

In [0]:
# New prompt explicitly requesting a single paragraph
paragraph_prompt = (
    "Based on the following sectional summaries, generate a single, "
    "coherent **paragraph summary** (no lists, no tables, and no bullet points) "
    "that provides a concise overview of Nvidia's history and key achievements.\n\n"
    f"SUMMARIES:\n\n{combined_text}"
)

final_response = chat_model.invoke(paragraph_prompt)

# Use the same extraction function to ensure you get the clean text string
paragraph_summary = extract_final_summary_text(final_response.content)

print(paragraph_summary)

In [0]:
import ast
from typing import Any

def extract_clean_text_from_object(parsed_object: Any) -> str:
    """
    Takes a PARSED Python list/dict object (not the raw string) and extracts 
    the final 'text' content, then cleans up Unicode characters.
    """
    raw_text = None
    
    # The parsed object is a list of dictionaries
    if isinstance(parsed_object, list):
        for item in parsed_object:
            # Look for the dictionary where the 'type' is 'text'
            if isinstance(item, dict) and item.get('type') == 'text' and 'text' in item:
                raw_text = item['text']
                break
            
    if raw_text is None:
        return "[ERROR: Could not find 'text' in the response structure.]"

    try:
        # Decode Unicode escapes for clean display
        cleaned_text = raw_text.encode('latin1').decode('unicode-escape')
        # Manual cleanup for common characters:
        cleaned_text = cleaned_text.replace('\u2011', '-') # Figure dash/hyphen
        cleaned_text = cleaned_text.replace('\u2022', '•') # Bullet point
        
        return cleaned_text.strip()
    except Exception:
        return raw_text.strip()

string_data = clean_paragraph_summary

try:
    parsed_content = ast.literal_eval(string_data)
except Exception as e:
    print(f"ERROR: Failed to parse string using ast.literal_eval: {e}")
    # Fallback: Print the raw string if parsing fails
    print(string_data)
    exit()

final_summary_paragraph = extract_clean_text_from_object(parsed_content)

print("📝 Final Summary:\n")
print(final_summary_paragraph)